# EMA200 touch bounce — IS/OOS validation

**Strategy (long-only, daily EOD — same definition as the scanner):**
- **Trend:** Price held above rising EMA200 for `min_days_above` sessions; EMA slope > floor.
- **Touch:** Session **low** reaches EMA200 + `touch_tolerance_atr` × ATR14.
- **Hold / rejection:** **Close** stays **≥** EMA200 (AMZN-style pullback; not a breakdown).
- **Trade sim:** Enter at signal close; exit on 2×ATR stop (low), 20-bar time stop, or close < EMA200.

**Research (TEMA/KAMA-shaped):**
- Event study: forward returns 1/5/10/20d, MAE/MFE
- Baselines: random days (regime-matched), EMA150/EMA250 touches
- Walk-forward: 60/40 IS/OOS + param grid stability on OOS Sharpe
- Monte Carlo bootstrap on trade returns

Intraday NQ / 15m AMZN charts are **Phase 3**; this notebook validates the **daily** hypothesis gate (Phase 2).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.ema200_scanner.config.loader import load_config
from pipeline.research.ema200_backtest import (
    DEFAULT_START,
    TRAIN_RATIO,
    analyze_ticker,
    benchmark_regime,
    default_param_grid,
    download_ohlcv,
    run_batch,
    write_report,
)

cfg = load_config()
TICKERS = ["AMZN", "AAPL", "SPY", "QQQ", "NVDA"]
print("Config mode:", cfg.get("mode"), "| EMA period:", cfg["ema_period"])
print("Tickers:", TICKERS, "| Start:", DEFAULT_START, "| IS ratio:", TRAIN_RATIO)

In [ ]:
spy = download_ohlcv("SPY", DEFAULT_START)
regime = benchmark_regime(spy["close"], int(cfg["ema_period"]), int(cfg["history_min_bars"]))
regime.tail()

In [ ]:
# Per-ticker analysis (downloads ~5y adjusted daily OHLCV)
results = []
for t in TICKERS:
    ohlcv = download_ohlcv(t, DEFAULT_START)
    # Compact grid in notebook for speed; use default_param_grid() for full sensitivity
    grid = [p for p in default_param_grid() if p.ema_period == 200]
    results.append(analyze_ticker(t, ohlcv, regime, grid=grid))

rows = []
for r in results:
    rows.append({
        "ticker": r.ticker,
        "events": r.n_events,
        "fwd5d_mean": r.event_stats.get("fwd_ret_5d_mean"),
        "is_sharpe": r.metrics["is"]["sharpe"],
        "oos_sharpe": r.metrics["oos"]["sharpe"],
        "bh_sharpe": r.metrics["buyhold"]["sharpe"],
        "mc_p50": r.monte_carlo.get("mean_return_p50"),
    })
summary = pd.DataFrame(rows)
summary

In [ ]:
report_path = ROOT / "pipeline" / "research" / "report.md"
write_report(results, report_path)
print("Wrote", report_path)

In [ ]:
# AMZN example: equity curve of long-only touch simulation (production params)
from pipeline.ema200_scanner.signals.ema200_touch import detect_events
from pipeline.research.ema200_backtest import cfg_touch_params, simulate_long_touch
from pipeline.research.kama_backtest import compute_metrics

amzn = download_ohlcv("AMZN", DEFAULT_START)
ev = detect_events(amzn, cfg_touch_params(), regime_series=regime)
trades, strat_rets = simulate_long_touch(amzn, ev)
eq = (1 + pd.Series(strat_rets, index=amzn.index[1:])).cumprod()
bh = (1 + amzn["close"].pct_change().fillna(0).iloc[1:]).cumprod()

fig, ax = plt.subplots(figsize=(10, 4))
eq.plot(ax=ax, label="EMA200 touch long sim")
bh.plot(ax=ax, label="Buy & hold", alpha=0.7)
ax.set_title("AMZN — daily EMA200 touch bounce simulation")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
print("Trades:", len(trades), "| Full Sharpe:", round(compute_metrics(strat_rets).sharpe, 3))

## Phase 2 gate checklist

1. **EMA200 vs EMA150/250:** Compare `fwd_ret_5d_mean` in `report.md` baselines — 200 should beat neighbors or document *no special level*.
2. **vs random:** Event 5d mean should beat regime-matched random entries.
3. **OOS Sharpe:** Strategy sim should not collapse vs IS (walk-forward table above).
4. **Monte Carlo:** If p5 of bootstrap mean return < 0, treat edge as fragile.
5. If gate fails → set `mode: watchlist_only` in YAML and skip signal alerts (Phase 3+).